# Image Classifier with CNN (Keras)

Flow: load train and test CSVs, sample rows, split label from pixels, reshape to 28x28x1, scale to 0 to 1, build the CNN, train 5 epochs, report test accuracy, show 5 test images with true and predicted labels.

Each CSV should have one label column and 784 pixel columns.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## 1. Upload your two CSV files
Run this cell and pick both files (train and test). If you prefer Google Drive, skip this cell and set the paths in the next cell to your Drive location after running `drive.mount('/content/drive')`.

## 2. Load and sample the data
Change the two file names below to match what you uploaded.

In [4]:
TRAIN_CSV = "fashion-mnist_train.csv"   # <-- change to your training file name
TEST_CSV  = "fashion-mnist_test.csv"    # <-- change to your testing file name
LABEL_COL = None          # set to the label column name, or leave None to use the first column

N_TRAIN = 5000
N_TEST = 2000

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print("Full train shape:", train_df.shape)
print("Full test shape:", test_df.shape)

# random samples
train_df = train_df.sample(n=min(N_TRAIN, len(train_df)), random_state=42).reset_index(drop=True)
test_df = test_df.sample(n=min(N_TEST, len(test_df)), random_state=42).reset_index(drop=True)
print("Sampled train shape:", train_df.shape)
print("Sampled test shape:", test_df.shape)
train_df.head()

Full train shape: (60000, 785)
Full test shape: (10000, 785)
Sampled train shape: (5000, 785)
Sampled test shape: (2000, 785)


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,8,0,0,0,0,0,0,0,0,0,...,187,190,196,201,209,221,218,211,102,0
2,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Separate label, reshape to 28x28x1, normalize

In [ ]:
label_col = LABEL_COL if LABEL_COL is not None else train_df.columns[0]
print("Label column:", label_col)

def prepare(df):
    y = df[label_col].values.astype("int64")
    X = df.drop(columns=[label_col]).values.astype("float32")
    assert X.shape[1] == 784, f"Expected 784 pixel columns, got {X.shape[1]}"
    X = X.reshape(-1, 28, 28, 1) / 255.0
    return X, y

X_train, y_train = prepare(train_df)
X_test, y_test = prepare(test_df)

n_classes = len(np.unique(np.concatenate([y_train, y_test])))
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Pixel range:", X_train.min(), "to", X_train.max())
print("Classes:", n_classes, np.unique(y_train))

## 4. Build the CNN

In [ ]:
# Class indices must run 0..n_classes-1 for sparse categorical loss
n_classes = int(max(y_train.max(), y_test.max())) + 1

model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(n_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 5. Train for 5 epochs

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1,
)

## 6. Test accuracy

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

## 7. Show 5 test images with true vs predicted labels

In [ ]:
# Optional: put your class names here (example for Fashion-MNIST), else numbers are shown
CLASS_NAMES = None
# CLASS_NAMES = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

def name(i):
    return CLASS_NAMES[i] if CLASS_NAMES else str(i)

idx = np.random.choice(len(X_test), 5, replace=False)
probs = model.predict(X_test[idx], verbose=0)
preds = probs.argmax(axis=1)

fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for ax, i, p in zip(axes, idx, preds):
    ax.imshow(X_test[i].squeeze(), cmap="gray")
    correct = (p == y_test[i])
    ax.set_title(f"True: {name(y_test[i])}\nPred: {name(p)}", color="green" if correct else "red", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()